# 🏗️ Starter — Neural Network Mini dari Nol (Bab 4)> Notebook eksperimen. **Prasyarat:** semua 48 test hijau> (`python -m unittest discover -s tests -v`).**Aturan main:** semua keputusan (arsitektur, lr, early stop) diambil di**validation set**. Test set hanya disentuh di Bagian 6, sekali.

## Setup & Peta Data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from nn_mini.data import muat_semua
from nn_mini.layers import Dense, Sequential
from nn_mini.train import train_model

X_train, y_train, X_val, y_val, X_test, y_test = muat_semua()
print('splits:', len(y_train), len(y_val), len(y_test),
      '| positif:', int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))


In [ ]:
# Lihat dulu bentuk datanya — non-linear seperti dua bulan yang mengait
plt.figure(figsize=(5, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=15, alpha=0.7)
plt.title('Dua bulan (train)'); plt.xlabel('x1'); plt.ylabel('x2'); plt.show()


## Bagian 1 — Baseline Linear DuluSebelum MLP, buktikan bahwa model LINEAR mentok. Gunakan `Sequential`dengan SATU layer output sigmoid tanpa hidden layer — itu logistic regression.

In [ ]:
linear_model = Sequential([Dense(1, activation='sigmoid', seed=0)])
hist_lin = train_model(linear_model, X_train, y_train, X_val, y_val,
                       epochs=400, batch_size=32, lr=0.5, seed=0)
print(f'Baseline linear: val acc = {hist_lin["val_acc"][-1]:.4f}')
print('-> Catat angka ini. Model non-linear nanti HARUS mengalahkannya.')


## Bagian 2 — MLP PertamamuArsitektur 16 → 8 → 1 (ReLU + ReLU + sigmoid), lr 0.1, batch 32,300 epoch tanpa early stopping. Bandingkan kurva loss-nya dengan baseline.

In [ ]:
mlp = Sequential([Dense(16, activation='relu', seed=7),
                  Dense(8, activation='relu', seed=8),
                  Dense(1, activation='sigmoid', seed=9)])
hist_mlp = train_model(mlp, X_train, y_train, X_val, y_val,
                       epochs=300, batch_size=32, lr=0.1, seed=0)
print(f'MLP: val acc = {hist_mlp["val_acc"][-1]:.4f}')

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_lin['train_loss'], label='linear train')
plt.plot(hist_lin['val_loss'], '--', label='linear val')
plt.plot(hist_mlp['train_loss'], label='MLP train')
plt.plot(hist_mlp['val_loss'], '--', label='MLP val')
plt.xlabel('epoch'); plt.ylabel('BCE'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist_lin['val_acc'], label='linear val acc')
plt.plot(hist_mlp['val_acc'], label='MLP val acc')
plt.axhline(0.5, color='gray', ls=':')
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.title('Akurasi'); plt.legend()
plt.tight_layout(); plt.show()


**Refleksi 1:** berapa % peningkatan val acc MLP atas baseline linear?Di kurva loss, MLP turun lebih dulu di bagian mana — awal atau akhir?

## Bagian 3 — Batas Keputusan (Visualisasi Paling Penting)

In [ ]:
# Gambar prediksi model di seluruh grid — lihat BATAS KEPUTUSANNYA.
def plot_boundary(model, judul):
    xs = np.linspace(X_train[:, 0].min() - 0.3, X_train[:, 0].max() + 0.3, 200)
    ys = np.linspace(X_train[:, 1].min() - 0.3, X_train[:, 1].max() + 0.3, 200)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    pp = model.predict_proba(grid).reshape(xx.shape)
    plt.contourf(xx, yy, pp, levels=20, cmap='coolwarm', alpha=0.6)
    plt.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap='coolwarm', s=12, edgecolors='k', lw=0.3)
    plt.title(judul)

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); plot_boundary(linear_model, 'Baseline linear — batas GARIS')
plt.subplot(1, 2, 2); plot_boundary(mlp, 'MLP — batas MELENGKUNG')
plt.tight_layout(); plt.show()


**Refleksi 2:** ini jawaban visual untuk pertanyaan analisis #1.Garis lurus tidak mungkin memisahkan dua bulan yang mengait; MLP bebasmelengkung karena hidden layer mengubah representasi fiturnya.

## Bagian 4 — Eksperimen Arsitektur (satu variabel per percobaan)

In [ ]:
# TODO: lengkapi eksperimen. Setiap baris = satu konfigurasi.
# Catat val acc tiap konfigurasi — cari pola, bukan cuma angka terbaik.
konfigurasi = [
    ('kecil (8)',        [Dense(8, activation='relu', seed=1),  Dense(1, activation='sigmoid', seed=2)]),
    ('sedang (16-8)',    [Dense(16, activation='relu', seed=7), Dense(8, activation='relu', seed=8), Dense(1, activation='sigmoid', seed=9)]),
    ('lebar (64)',       [Dense(64, activation='relu', seed=3), Dense(1, activation='sigmoid', seed=4)]),
    ('dalam (16-16-8)',  [Dense(16, activation='relu', seed=5), Dense(16, activation='relu', seed=6), Dense(8, activation='relu', seed=10), Dense(1, activation='sigmoid', seed=11)]),
]

# for nama, layers in konfigurasi:
#     model = Sequential(layers)
#     h = train_model(model, X_train, y_train, X_val, y_val,
#                     epochs=250, batch_size=32, lr=0.1, seed=0)
#     print(f'{nama:>16}: val acc = {h["val_acc"][-1]:.4f}  val loss = {h["val_loss"][-1]:.4f}')


**Refleksi 3:** dataset ini "mudah" (600 titik, 2 fitur) — arsitektur besartidak selalu menang. Di data apa model besar justru berisiko? (ingat Bab 3: overfitting)

## Bagian 5 — Learning Rate & Early Stopping

In [ ]:
# TODO: jalankan MLP yang sama dengan lr = 50.0 — amati kurva loss.
# hist_bad = train_model(..., lr=50.0, epochs=150)
# plt.plot(hist_bad['train_loss']); plt.title('lr=50: apa yang terjadi?'); plt.show()

# TODO: ulangi dengan early stopping patience=30 (epochs=1000).
# Berapa epoch yang benar-benar berjalan? Bandingkan val loss akhirnya
# dengan MLP tanpa early stopping (300 epoch) di Bagian 2.
# hist_es = train_model(..., epochs=1000, patience=30)
# print(len(hist_es['val_loss']), 'epoch berjalan')


## Bagian 6 — Sentuh Test Set (SEKALI!)

In [ ]:
# Pakai konfigurasi TERBAIK dari eksperimen val (Bagian 2 & 4).
p_test_lin = linear_model.predict_proba(X_test)
p_test_mlp = mlp.predict_proba(X_test)
print(f'test acc baseline linear : {((p_test_lin >= 0.5) == y_test).mean():.4f}')
print(f'test acc MLP             : {((p_test_mlp >= 0.5) == y_test).mean():.4f}')
print('-> Selisih ini yang kamu "jual" kalau ini proyek produksi.')


---## 📝 Pertanyaan Analisis (jawab dengan markdown, kata-kata sendiri + angka)1. Kenapa model linear mentok di 0.889 padahal dilatih 400 epoch?2. Jelaskan backpropagation dengan kata-katamu sendiri (gradien mengalir mundur).3. Saat lr=50, apa yang terlihat di kurva loss dan kenapa (mekanisme update)?4. Kenapa early stopping memantau val loss, bukan train loss? Bahaya patience terlalu kecil?5. Init semua bobot nol: apa yang terjadi di hidden layer (hint: simetri gradien)?Setelah semua terjawab → isi `RUBRIK.md`. Baru bandingkan dengan `solusi.ipynb`.